In [1]:
import sys
sys.path.append('..')
import module

In [ ]:
import os
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


class StderrFilter:
    """Filter untuk menekan log C++ dari MediaPipe/TensorFlow."""

    def __init__(self, original_stderr):
        self.original_stderr = original_stderr

    def write(self, message):
        suppressed_patterns = [
            "FaceBlendshapesGraph acceleration to xnnpack",
            "Feedback manager requires a model",
            "Sets FaceBlendshapesGraph",
            "Created TensorFlow Lite XNNPACK delegate",
        ]
        if message.strip() == "":
            return
        if any(pattern in message for pattern in suppressed_patterns):
            return
        self.original_stderr.write(message)

    def flush(self):
        self.original_stderr.flush()


sys.stderr = StderrFilter(sys.stderr)
warnings.filterwarnings('ignore')

W0000 00:00:1773928341.523000   72089 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773928341.536132   72092 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773928341.546421   72103 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
import os
from pathlib import Path


BASE_DATASOURCE_PATH=os.path.join(Path.home().as_posix(), "datasets", "anxiety_raw")
BASE_ANNOTATION_PATH=os.path.join(BASE_DATASOURCE_PATH, "annotations-v2.csv")

assert os.path.exists(BASE_DATASOURCE_PATH), f"Path tidak ditemukan: {BASE_DATASOURCE_PATH}"
assert os.path.exists(BASE_ANNOTATION_PATH), f"Path tidak ditemukan: {BASE_ANNOTATION_PATH}"

print("Datasource path successfully set.")

Datasource path successfully set.


In [4]:
import re
import pandas as pd
import numpy as np


df_annotation = pd.read_csv(BASE_ANNOTATION_PATH)
df_annotation.head()

,subject_id,subject_name,clip,stage,anxiety_level,anxiety_score,anxiety_category,gender,age,npy_path,video_path,frame_count
0,m__firmansyah,M Firmansyah,q3,before,high,0,NaN,man,0,/home/inadio/datasets/anxiety_raw/.cache_npy/b...,/home/inadio/datasets/anxiety_raw/before/anxie...,561
1,m__firmansyah,M Firmansyah,q4,before,high,0,NaN,man,0,/home/inadio/datasets/anxiety_raw/.cache_npy/b...,/home/inadio/datasets/anxiety_raw/before/anxie...,377
2,m__firmansyah,M Firmansyah,q5,before,high,0,NaN,man,0,/home/inadio/datasets/anxiety_raw/.cache_npy/b...,/home/inadio/datasets/anxiety_raw/before/anxie...,283
3,m__firmansyah,M Firmansyah,q1,before,high,0,NaN,man,0,/home/inadio/datasets/anxiety_raw/.cache_npy/b...,/home/inadio/datasets/anxiety_raw/before/anxie...,854
4,m__firmansyah,M Firmansyah,q2,before,high,0,NaN,man,0,/home/inadio/datasets/anxiety_raw/.cache_npy/b...,/home/inadio/datasets/anxiety_raw/before/anxie...,338


In [5]:
import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from src.apex.modules.v2 import ApexPhaseSpotter


spotter = ApexPhaseSpotter()

for index, row in tqdm(df_annotation.iterrows(), total=len(df_annotation), desc="Processing Videos Serial"):
    subject_name = row["subject_name"]
    anxiety_level = row["anxiety_level"]
    question_number = Path(row["video_path"]).parents[0].name

    if os.path.exists(row["npy_path"]):
        continue

    try:
        spotter.process(row["video_path"])
        flow_data = spotter.export_flow_data()
        
        data = {
            "frames": flow_data["frames"],
            "magnitudes": flow_data["magnitudes"],
            "frame_count": flow_data["frame_count"],
            "landmark_detection_rate": flow_data["landmark_detection_rate"],
            "roi_order": flow_data["roi_order"],
        }

        np.save(row["npy_path"], data, allow_pickle=True)
        
    except Exception as e:
        tqdm.write(f"ERROR memproses video {row['video_path']}: {e}")
        
    finally:
        # Hapus data sementara sisa proses & jalankan Garbage Collector 
        if 'flow_data' in locals(): del flow_data
        if 'data' in locals(): del data
        gc.collect()

if hasattr(spotter, 'tvl1') and spotter.tvl1:
    spotter.tvl1.close()
    


Processing Videos Serial:   0%|          | 0/560 [00:00<?, ?it/s]

ERROR memproses video /home/inadio/datasets/anxiety_raw/after/tidak/fandy_wahyu_hanzura_1765270992417/q4/answer_4_497eb431-dd00-4530-858f-e8ec710014e7_sec.avi: No face landmarks detected.


Load data yang sudah tersimpan dan menyimpan ke file csv

In [7]:
import json

# Pastikan kolom target ada
target_cols = [
    "frame_count",
    "landmark_detection_rate",
    "roi_order",
]
for col in target_cols:
    if col not in df_annotation.columns:
        df_annotation[col] = None

for idx, row in tqdm(df_annotation.iterrows(), total=len(df_annotation), desc="Updating annotation from NPY"):
    npy_path = row["npy_path"]

    if not os.path.exists(npy_path):
        df_annotation.at[idx, "is_valid"] = False
        tqdm.write(f"WARNING: NPY tidak ditemukan: {npy_path}")
        continue

    try:
        data = np.load(npy_path, allow_pickle=True).item()

        df_annotation.at[idx, "is_valid"] = True
        df_annotation.at[idx, "frame_count"] = data.get("frame_count", None)
        df_annotation.at[idx, "landmark_detection_rate"] = data.get("landmark_detection_rate", None)

        roi_order = data.get("roi_order", None)
        df_annotation.at[idx, "roi_order"] = json.dumps(roi_order) if roi_order is not None else None

    except Exception as e:
        df_annotation.at[idx, "is_valid"] = False
        tqdm.write(f"ERROR memuat NPY {npy_path}: {e}")

# Simpan hasil update anotasi
df_annotation.to_csv(BASE_ANNOTATION_PATH, index=False)
print(f"Updated annotation saved to: {BASE_ANNOTATION_PATH}")

Updating annotation from NPY:   0%|          | 0/560 [00:00<?, ?it/s]

Updated annotation saved to: /home/inadio/datasets/anxiety_raw/annotations-v2.csv
